# 02 - Fine-tune transformers

**Goal:** fine-tune MuRIL (and optionally mBERT) and compare against the baseline.

In [1]:
%pip install -q ftfy tweet-preprocessor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, subprocess

REPO_URL = "https://github.com/23f2004336/hinglish-sentiment.git"

def has_src(base):
    return os.path.exists(os.path.join(base, "src", "data.py"))

if has_src(".."):
    sys.path.insert(0, "..")
    RESULTS_DIR = "../results"
elif has_src("."):
    sys.path.insert(0, ".")
    RESULTS_DIR = "results"
else:
    if not os.path.exists("hinglish-sentiment"):
        subprocess.run(["git", "clone", REPO_URL], check=True)   # needs Internet ON in Kaggle
    sys.path.insert(0, os.path.abspath("hinglish-sentiment"))
    RESULTS_DIR = "."

Cloning into 'hinglish-sentiment'...


In [3]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

In [4]:
import torch
from src.data import load_data
from src.train import run

HF_USER = "Shrishti03"      
OUT_ROOT = "/kaggle/working" if os.path.exists("/kaggle") else "../outputs"
os.makedirs(OUT_ROOT, exist_ok=True)

print("GPU available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
dfs = load_data()

GPU available: True | devices: 2


README.md:   0%|          | 0.00/533 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 22.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.92MB            

...

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.98MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/96761 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8538 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8538 [00:00<?, ? examples/s]

## MuRIL

In [5]:
SMOKE = False    

res_muril = run(
    dfs,
    name="google/muril-base-cased",
    tag="muril",
    epochs=1 if SMOKE else 2,
    out_root=OUT_ROOT,
    repo=f"{HF_USER}/hinglish-sentiment-smoke-test" if SMOKE else f"{HF_USER}/hinglish-sentiment-muril",
    max_train=3000 if SMOKE else None,
)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

## mBERT

In [6]:
RUN_MBERT = False
if RUN_MBERT:
    res_mbert = run(dfs, name="bert-base-multilingual-cased", tag="mbert", epochs=3, out_root=OUT_ROOT)

## Results table

In [7]:
import pandas as pd

rows = {
    "TF-IDF + LogReg": {"overall": 0.817, "hinglish": 0.518, "english": 0.826},
    "mBERT (3 epochs)": {"overall": 0.847, "hinglish": 0.630, "english": 0.855},   # from the first run
    "MuRIL": {k: res_muril[k] for k in ["overall", "hinglish", "english"]},
}
table = pd.DataFrame(rows).T.round(3)
table

,overall,hinglish,english
TF-IDF + LogReg,0.817,0.518,0.826
mBERT (3 epochs),0.847,0.630,0.855
MuRIL,0.319,0.121,0.326
